## Tactical Development


### Dataset :: (fake dataset - databricks market place)



- "databricks_simulated_retail_customer_data"

1) The first step is to explore the dataset as much as possible, to identify all the possible relationships existent between the tables and volumes;

2) Ideally more than one architecture for data modeling, until "best" pipeline and insight is found;


## V01

### New tables

##### creating table from VOLUME

In [0]:
# %sql
# Sample Script 
# CREATE SCHEMA IF NOT EXISTS medallion_catalog.db_bronze
# MANAGED LOCATION 'abfss://catalog@container.dfs.core.windows.net/folder';

# USE CATALOG medallion_catalog;
# USE SCHEMA db_bronze;

# DROP TABLE IF EXISTS medallion_catalog.db_bronze.companies;

# CREATE TABLE IF NOT EXISTS medallion_catalog.db_bronze.companies 
# AS
# SELECT company_name, founded_date, country
#   FROM read_files('/Volumes/medallion_catalog/db_landing/vol_medallion/top_tech_companies/',
#   format => 'csv',
#   header => true);


#### creating tables from tables

In [0]:
# %sql
# -- 1. Create the table structure with an IDENTITY column
# CREATE TABLE IF NOT EXISTS main.default.user_profile (
#   user_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
#   user_name STRING,
#   signup_date DATE,
#   account_balance DECIMAL(18, 2),
#   is_active BOOLEAN
# )
# USING DELTA;

# -- 2. Populate the table (excluding the identity column)
# INSERT INTO main.default.user_profile (user_name, signup_date, account_balance, is_active)
# SELECT 
#   name, 
#   CAST(registration_time AS DATE), 
#   CAST(balance AS DECIMAL(18, 2)), 
#   true
# FROM main.staging.raw_users;

## Tables

### Customers ::

##### Table Customers address ::

In [0]:
%sql
SELECT
customer_id,
state,
city,
postcode,
street,
number,
unit,
region,
district
from databricks_simulated_retail_customer_data.v01.customers limit 5


In [0]:
%sql
-- 1. Define the table structure with types and Identity column
CREATE OR REPLACE TABLE end_to_end_retail.db_landing.customer_address (
  -- customer_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
  customer_id STRING,
  state STRING,
  city STRING,
  postcode STRING,
  street STRING,
  number STRING,
  unit STRING,
  region STRING,
  district STRING
)
USING DELTA;

-- 2. Insert data from your source (omit customer_id to let it auto-generate)
INSERT INTO end_to_end_retail.db_landing.customer_address (
  customer_id,
  state, 
  city, 
  postcode, 
  street, 
  number, 
  unit, 
  region, 
  district
)
SELECT
  customer_id, 
  state, 
  city, 
  postcode, 
  street, 
  number, 
  unit, 
  region, 
  district
FROM databricks_simulated_retail_customer_data.v01.customers;

##### Table Customers geo_loc ::

In [0]:
 %sql
SELECT 
 customer_id,
 lon,
 lat,
 ship_to_address,
 valid_from,
 valid_to
FROM databricks_simulated_retail_customer_data.v01.customers;

In [0]:
 %sql
 CREATE OR REPLACE TABLE end_to_end_retail.db_landing.customer_geoloc (
 customer_id STRING,
 lon STRING,
 lat STRING,
 ship_to_address STRING,
 valid_from STRING,
 valid_to STRING
)
USING DELTA;

 -- 2. Insert data from your source (omit customer_id to let it auto-generate)
INSERT INTO end_to_end_retail.db_landing.customer_geoloc (
 customer_id,
 lon,
 lat,
 ship_to_address,
 valid_from,
 valid_to
)
SELECT 
 customer_id,
 lon,
 lat,
 ship_to_address,
 valid_from,
 valid_to
FROM databricks_simulated_retail_customer_data.v01.customers;

##### Table Customers tax_id ::

In [0]:
%sql
select 
customer_id,
customer_name,
tax_id,
tax_code,
units_purchased,
loyalty_segment
from databricks_simulated_retail_customer_data.v01.customers limit 5

In [0]:
%sql
CREATE OR REPLACE TABLE end_to_end_retail.db_landing.customer_taxid (
customer_id STRING,
customer_name STRING,
tax_id STRING,
tax_code STRING,
units_purchased STRING,
loyalty_segment STRING
)
USING DELTA;

 -- 2. Insert data from your source (omit customer_id to let it auto-generate)
INSERT INTO end_to_end_retail.db_landing.customer_taxid (
customer_id,
customer_name,
tax_id,
tax_code,
units_purchased,
loyalty_segment
)
select 
customer_id,
customer_name,
tax_id,
tax_code,
units_purchased,
loyalty_segment
from databricks_simulated_retail_customer_data.v01.customers;

### Sales ::

In [0]:
%sql
SELECT
    customer_id,
    customer_name,
    order_date,
    product_category,
    total_price,
    -- ai_query("databricks-meta-llama-3-3-70b-instruct", 'Apply NLP to extract only meaningful description 5 word max: ' || product_name) as response
    -- ai_query("databricks-meta-llama-3-3-70b-instruct", 'Summarize the product name only 5 words:' || product_name) as response,
    product_name,
    get_json_object(product, '$.curr') AS curr,
    get_json_object(product, '$.id') AS id,
    get_json_object(product, '$.name') AS product_name,
    get_json_object(product, '$.price') AS price,
    get_json_object(product, '$.qty') AS qty,
    get_json_object(product, '$.unit') AS unit
FROM (
  SELECT * FROM databricks_simulated_retail_customer_data.v01.sales
) t
limit 5

In [0]:
%sql
CREATE OR REPLACE TABLE end_to_end_retail.db_landing.sales (
customer_id STRING,
customer_name STRING,
order_date DATE,
product_category STRING,
total_price LONG,
product_name STRING,
curr STRING,
id STRING,
product STRING,
price STRING,
qty STRING,
unit STRING
)
USING DELTA;

 -- 2. Insert data from your source (omit customer_id to let it auto-generate)
INSERT INTO end_to_end_retail.db_landing.sales (
customer_id,
customer_name,
order_date,
product_category,
total_price,
product_name,
curr,
id,
product,
price,
qty,
unit
)
SELECT
    customer_id,
    customer_name,
    order_date,
    product_category,
    total_price,
    -- ai_query("databricks-meta-llama-3-3-70b-instruct", 'Apply NLP to extract only meaningful description 5 word max: ' || product_name) as response
    -- ai_query("databricks-meta-llama-3-3-70b-instruct", 'Summarize the product name only 5 words:' || product_name) as response,
    product_name,
    get_json_object(product, '$.curr') AS curr,
    get_json_object(product, '$.id') AS id,
    get_json_object(product, '$.name') AS product,
    get_json_object(product, '$.price') AS price,
    get_json_object(product, '$.qty') AS qty,
    get_json_object(product, '$.unit') AS unit
FROM (
  SELECT * FROM databricks_simulated_retail_customer_data.v01.sales
) t



### Sales_order ::

In [0]:
%sql
select 
customer_id,
customer_name,
number_of_line_items,
order_datetime,
order_number,
ordered_products,
promo_info,
clicked_items
FROM databricks_simulated_retail_customer_data.v01.sales_orders
limit 10

#### Table Clicked Items

In [0]:
%sql
  SELECT
    customer_id,
    customer_name,
    number_of_line_items,
    order_datetime,
    order_number,
    ordered_products,
    promo_info,
    clicked_items
  FROM databricks_simulated_retail_customer_data.v01.sales_orders limit 1

In [0]:
%sql
select 
customer_id,
clicked_items
FROM databricks_simulated_retail_customer_data.v01.sales_orders
where customer_id = '14939501'

In [0]:
%sql
CREATE OR REPLACE TABLE end_to_end_retail.db_landing.clicked_items (
customer_id STRING,
clicked_items STRING
)
USING DELTA;

 -- 2. Insert data from your source (omit customer_id to let it auto-generate)
INSERT INTO end_to_end_retail.db_landing.clicked_items (
select 
customer_id,
clicked_items

FROM (
  SELECT
    customer_id,
    clicked_items
  FROM databricks_simulated_retail_customer_data.v01.sales_orders
) t
)

In [0]:
%sql
SELECT * FROM databricks_simulated_retail_customer_data.v01.sales_orders limit 1

#### Table Promo info

In [0]:
%sql
SELECT
  customer_id,
  order_number,
  promo_info
FROM (
  SELECT
    customer_id,
    order_number,
    promo_info
  FROM databricks_simulated_retail_customer_data.v01.sales_orders
  WHERE promo_info IS NOT NULL
) t

In [0]:
%sql
CREATE OR REPLACE TABLE end_to_end_retail.db_landing.promo_info (
  customer_id STRING,
  order_number STRING,
  promo_info STRING
)
USING DELTA;

-- 2. Insert data from your source
INSERT INTO end_to_end_retail.db_landing.promo_info (
  customer_id,
  order_number,
  promo_info
)
SELECT
  customer_id,
  order_number,
  promo_info
  FROM databricks_simulated_retail_customer_data.v01.sales_orders  
  WHERE promo_info IS NOT NULL

#### Table Ordered products

In [0]:
%sql
SELECT
  customer_id,
  order_number,
  ordered_products

FROM (
  SELECT
    customer_id,
    order_number,
    ordered_products
  FROM databricks_simulated_retail_customer_data.v01.sales_orders
) t
limit 1

In [0]:
%sql
CREATE OR REPLACE TABLE end_to_end_retail.db_landing.ordered_product (
  customer_id STRING,
  order_number STRING,
  ordered_products STRING
)
USING DELTA;

-- 2. Insert data from your source
INSERT INTO end_to_end_retail.db_landing.ordered_product (
  customer_id,
  order_number,
  ordered_products
)
SELECT
  customer_id,
  order_number,
  ordered_products
  FROM databricks_simulated_retail_customer_data.v01.sales_orders